# <font color=firebrick>Econometrics — M2 MBFA</font>
## TD 1 / 6 — Applying OLS: Properties and Dataset

**Prerequisites:** Sessions 1-2 (CM) — OLS estimator, variance, and t-test (one regressor, no intercept).

**Format:** two exercises (Parts 1-2) + an applied exercise on a real dataset (Part 3).

**Today's objectives:**
- Practice the properties of expectation and variance underlying the OLS results.
- Use simulation to *check* that $\hat\beta$ is unbiased and that its variance matches the formula from Session 2.
- Apply OLS and t-test to real data, and learn to separate a statistical conclusion ("is this effect real, or noise?") from an economic one ("does this effect matter in practice?").

We'll work with the dataset `ceosal1` (CEO salary and firm performance) from the `wooldridge` Python package.

---


## Part 1 — Properties of mean and variance

Before we start looking at a database, let’s make sure we know what expected value and variance are, and how they behave under simple transformations. We’ll figure out each of these properties using numerical calculations: we’ll simulate a large number of samples, work out the sample means and variances, and then compare them to the theoretical values.


**Task 1.1 - Part A (by hand).** Consider two **independent** random variables with simple discrete distributions:
- $X = 10$ with probability $0.5$, $X=20$ with probability $0.5$;
- $Y = 5$ with probability $0.5$, $Y=15$ with probability $0.5$, independent of $X$.

1. Compute $\mathbb{E}[X]$ and $\mathbb{E}[Y]$ using $\mathbb{E}[X]=\sum_x x\cdot P(X=x)$.
2. List the 4 equally likely outcomes of the pair $(X,Y)$ and compute $\mathbb{E}[X+Y]$ **directly** from them. Compare it to $\mathbb{E}[X]+\mathbb{E}[Y]$.
3. Compute $\text{Var}(X)$ and $\text{Var}(Y)$ using $\text{Var}(X)=\sum_x (x-\mathbb{E}[X])^2\cdot P(X=x)$.
4. Using the same 4 outcomes as in Q2, compute $\text{Var}(X+Y)$ **directly**. Compare it to $\text{Var}(X)+\text{Var}(Y)$.

<details>
<summary>Show answer</summary>

1. $\mathbb{E}[X]=0.5(10)+0.5(20)=15$. $\mathbb{E}[Y]=0.5(5)+0.5(15)=10$.

2. The 4 equally likely outcomes of $X+Y$ are $15,25,25,35$ (from $(10,5),(10,15),(20,5),(20,15)$), so $\mathbb{E}[X+Y]=\frac{15+25+25+35}{4}=25$. Indeed $\mathbb{E}[X]+\mathbb{E}[Y]=15+10=25$: they match.

3. $\text{Var}(X)=0.5(10-15)^2+0.5(20-15)^2=0.5(25)+0.5(25)=25$. Similarly $\text{Var}(Y)=0.5(5-10)^2+0.5(15-10)^2=25$.

4. The 4 values of $X+Y$ found in Q2 have mean $25$; their variance is $\frac{1}{4}\big[(15-25)^2+(25-25)^2+(25-25)^2+(35-25)^2\big]=\frac{100+0+0+100}{4}=50$. Indeed $\text{Var}(X)+\text{Var}(Y)=25+25=50$: they match, because $X$ and $Y$ are independent.

</details>

**Task 1.1 - Part B (computational, at scale).** The calculation above uses only 4 possible outcomes - small enough to do by hand, but also small enough that it could be a coincidence. Let's now check that the same two properties hold with a much larger, continuously-distributed pair $(X,Y)$, by simulating $n\_draws=100{,}000$ independent draws of each and comparing sample moments to their theoretical values.


In [2]:
import numpy as np

rng = np.random.default_rng(seed=42)

n_draws = 100_000  # a large number of draws, so sample moments are close to their theoretical values

X = rng.normal(loc=5, scale=2, size=n_draws)   # X ~ N(5, 2^2):  E[X]=5, Var(X)=4
Y = rng.normal(loc=3, scale=1, size=n_draws)   # Y ~ N(3, 1^2):  E[Y]=3, Var(Y)=1, independent of X

print("Sample mean of X:", X.mean(), " (theory: 5)")
print("Sample variance of X:", X.var(), " (theory: 4)")


Sample mean of X: 4.991534139442154  (theory: 5)
Sample variance of X: 4.029681278474262  (theory: 4)


Complete the code cell below to verify the same two properties, this time with a large simulated `X` and `Y`:
1. $\mathbb{E}[X+Y] = \mathbb{E}[X]+\mathbb{E}[Y]$
2. $\text{Var}(X+Y) = \text{Var}(X)+\text{Var}(Y)$ (true here because $X$ and $Y$ are independent)


In [6]:
# TODO: compute the sample mean of X + Y, and compare it to X.mean() + Y.mean()
mean_XY = ...
print("Mean of X+Y:      ", mean_XY)
print("E[X] + E[Y]:      ", X.mean() + Y.mean())

# TODO: compute the sample variance of X + Y, and compare it to X.var() + Y.var()
var_XY = ...
print("Var(X+Y):         ", var_XY)
print("Var(X) + Var(Y):  ", X.var() + Y.var())


Mean of X+Y:       Ellipsis
E[X] + E[Y]:       7.994251410558271
Var(X+Y):          Ellipsis
Var(X) + Var(Y):   5.03326203490244


**Task 1.2 - Part A (by hand).** Reuse the distribution of $X$ from Task 1.1 ($X=10$ or $20$, each with probability $0.5$; $\mathbb{E}[X]=15$, $\text{Var}(X)=25$). Let $a=3$.

1. List the 2 possible values of $aX$, and compute $\mathbb{E}[aX]$ and $\text{Var}(aX)$ directly from them.
2. Compare $\text{Var}(aX)$ to $a^2\,\text{Var}(X)$.

<details>
<summary>Show answer</summary>

$aX\in\{30,60\}$, each with probability $0.5$, so $\mathbb{E}[aX]=45$ and $\text{Var}(aX)=0.5(30-45)^2+0.5(60-45)^2=0.5(225)+0.5(225)=225$.

$a^2\,\text{Var}(X)=9\times 25=225$: they match.

</details>

**Task 1.2 - Part B (computational).** Now check the same scaling property $\text{Var}(aX)=a^2\text{Var}(X)$ for $a=3$, on the large simulated `X`. Complete the code below.


In [7]:
a = 3

# TODO: compute the sample variance of a*X, and compare it to a**2 * X.var()
var_aX = ...
print("Var(a*X):        ", var_aX)
print("a^2 * Var(X):    ", a**2 * X.var())


Var(a*X):         Ellipsis
a^2 * Var(X):     36.26713150626836


**Task 1.3 (no code needed).** The property $\text{Var}(X+Y)=\text{Var}(X)+\text{Var}(Y)$ relied on $X$ and $Y$ being *independent*. Suppose instead $X$ and $Y$ were two stock returns that tend to move together (positively correlated). Would you expect $\text{Var}(X+Y)$ to be larger or smaller than $\text{Var}(X)+\text{Var}(Y)$? (Just the intuition - no formula needed.)

<details>
<summary>Show answer</summary>

Larger. When $X$ and $Y$ move together, their fluctuations reinforce each other instead of partly cancelling out. The exact formula is $\text{Var}(X+Y)=\text{Var}(X)+\text{Var}(Y)+2\,\text{Cov}(X,Y)$, and the extra covariance term is positive here. This is exactly why diversifying a portfolio with *uncorrelated* (or negatively correlated) assets reduces risk more than combining assets that move together.

</details>


**Task 1.4 - Part A (by hand).** Take a tiny version of the Session-1 DGP $y=\beta x+\varepsilon$ with $\beta=2$. Let $x\in\{1,3\}$, each with probability $0.5$, and (independently) $\varepsilon\in\{-1,1\}$, each with probability $0.5$.

1. List the 4 equally likely combinations of $(x,\varepsilon)$ and compute $y=\beta x+\varepsilon$ for each.
2. Compute $\mathbb{E}[y]$ and $\text{Var}(y)$ directly from these 4 values.
3. Separately, compute $\text{Var}(x)$ and $\text{Var}(\varepsilon)$, and check that $\text{Var}(y)=\beta^2\text{Var}(x)+\text{Var}(\varepsilon)$.

<details>
<summary>Show answer</summary>

The 4 combinations $(x,\varepsilon)$ give $y=2x+\varepsilon \in \{1,\,3,\,5,\,7\}$ (from $(1,-1),(1,1),(3,-1),(3,1)$), each with probability $0.25$.

$\mathbb{E}[y]=\frac{1+3+5+7}{4}=4$. $\text{Var}(y)=\frac{(1-4)^2+(3-4)^2+(5-4)^2+(7-4)^2}{4}=\frac{9+1+1+9}{4}=5$.

$\text{Var}(x)=0.5(1-2)^2+0.5(3-2)^2=1$. $\text{Var}(\varepsilon)=0.5(-1)^2+0.5(1)^2=1$.

$\beta^2\text{Var}(x)+\text{Var}(\varepsilon)=4(1)+1=5$: matches $\text{Var}(y)=5$.

</details>

**Task 1.4 - Part B (computational).** Recall the Data-Generating Process from Session 1, $y=\beta x+\varepsilon$, and Task #1's result: $\text{Var}(y)=\beta^2\text{Var}(x)+\text{Var}(\varepsilon)$. Let's verify this at scale, on the very same simulated dataset used in Sessions 1-2.


In [ ]:
rng = np.random.default_rng(seed=10)
beta = 1
N = 100
x = rng.exponential(1, size=N)
eps = rng.normal(0, 1, size=N)
y = x * beta + eps

# TODO: compute Var(y) directly, and compare it to beta**2 * Var(x) + Var(eps)
var_y_direct = ...
var_y_formula = ...

print("Var(y), computed directly:                    ", var_y_direct)
print("beta^2 * Var(x) + Var(eps), from the formula: ", var_y_formula)


## Part 2 — Unbiasedness and variance formula

In Session 2 you *proved analytically* that (i) $\hat\beta$ is unbiased, $\mathbb{E}[\hat\beta]=\beta$, and (ii) $\text{Var}(\hat\beta)=\sigma^2/\sum_i x_i^2$. Let's now *check* both results with a simulation - the same tool you used in Session 1, applied to the estimator itself.

**Important design point.** The variance formula treats the $x_i$ as fixed. To test it properly, we therefore draw $x$ **once** and keep it fixed, and only re-draw the noise $\varepsilon$ many times - each repetition mimics "the same firms/individuals, but a new realization of luck/noise". If we re-drew $x$ every time as well, we would no longer be isolating the same source of variation as in the formula.

**Task 2.0 - Part A (by hand).** Before simulating thousands of samples, compute $\hat\beta$ once, by hand, on a tiny sample. Take $\beta=1$, $x=(1,2,3)$, and one specific realization of the noise $\varepsilon=(1,-1,2)$.

1. Compute $y_i=\beta x_i+\varepsilon_i$ for each $i$.
2. Using the closed-form estimator from Session 2, $\hat\beta=\dfrac{\sum_i x_iy_i}{\sum_i x_i^2}$, compute $\hat\beta$ for this sample.
3. Is $\hat\beta$ exactly equal to the true $\beta=1$? Should it be, based on a *single* sample?

<details>
<summary>Show answer</summary>

$y=(1\times1+1,\; 1\times2-1,\; 1\times3+2)=(2,1,5)$.

$\sum_i x_iy_i = 1(2)+2(1)+3(5)=2+2+15=19$, and $\sum_i x_i^2=1+4+9=14$, so $\hat\beta=19/14\approx1.36$.

No - $\hat\beta\approx1.36\neq1=\beta$. This is expected: with a single sample, the noise $\varepsilon$ does not average out, so $\hat\beta$ will almost never equal $\beta$ exactly. Unbiasedness is a statement about the *average* of $\hat\beta$ over many repeated samples, not about any one sample - which is exactly why we need Part B below.

</details>

**Task 2.0 - Part B (computational).** Complete the two missing lines below: the DGP from Session 1, and the closed-form OLS estimator from Session 2 - then repeat the whole draw-and-estimate process $B=2000$ times, to see what happens *on average*.


In [ ]:
rng = np.random.default_rng(seed=123)

beta_true = 1.0
sigma2 = 1.0     # true variance of the error term
N = 100          # sample size (fixed across all simulations)
B = 2000         # number of Monte Carlo repetitions

x = rng.exponential(1, size=N)   # x is drawn ONCE and held fixed across all B repetitions

beta_hats = np.zeros(B)

for b in range(B):
    eps_b = rng.normal(0, np.sqrt(sigma2), size=N)
    y_b = ...                          # TODO: recall the DGP from Session 1 (y = beta*x + eps)
    beta_hat_b = ...                   # TODO: recall the closed-form OLS estimator from Session 2
    beta_hats[b] = beta_hat_b

print("Average of the", B, "simulated beta_hat's: ", beta_hats.mean())
print("True beta:                                  ", beta_true)
print()
print("Empirical variance of beta_hat across simulations:", beta_hats.var())
print("Theoretical variance sigma^2 / sum(x^2):          ", sigma2 / np.sum(x**2))


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(beta_hats, bins=40, color='steelblue', alpha=0.7)
ax.axvline(beta_true, color='red', linewidth=2, label=fr'true $\beta={beta_true}$')
ax.axvline(beta_hats.mean(), color='black', linestyle='--', linewidth=2, label=r'average $\hat\beta$ across simulations')
ax.set_xlabel(r'$\hat\beta$')
ax.set_ylabel('count')
ax.set_title(fr'Distribution of $\hat\beta$ across {B} simulated samples (N={N} each)')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()


**Task 2.1.** Are the empirical average of $\hat\beta$ and the true $\beta$ close? What does this confirm?

**Task 2.2.** Are the empirical variance and the theoretical variance $\sigma^2/\sum_i x_i^2$ close? What does this confirm?

**Task 2.3 (an important distinction).** Two different numbers in this simulation can each be made larger: $N$ (the size of *each* simulated sample) and $B$ (the *number of times* we repeat the simulation).
- Increase $N$ only (keep $B$ fixed) and re-run. What happens to the spread of the histogram?
- Increase $B$ only (keep $N$ fixed) and re-run. What happens to the spread of the histogram? What changes instead?

<details>
<summary>Show answer</summary>

Increasing **$N$** genuinely makes $\hat\beta$ more precise: with more data per sample, $\sum_i x_i^2$ grows, so $\text{Var}(\hat\beta)=\sigma^2/\sum_i x_i^2$ shrinks - the histogram gets narrower.

Increasing **$B$** does not change the real precision of $\hat\beta$ at all (each simulated sample still has the same $N$) - it only makes *our estimate* of the histogram/variance more accurate, because we are averaging over more repetitions. With a small $B$, the histogram looks noisy and the reported empirical variance bounces around from run to run; with a large $B$, it stabilizes. $B$ is a feature of *our simulation*, not of the real world - in real life we only ever observe **one** sample of size $N$, never $B$ repetitions of it.

</details>

---


## Part 3 — Applying OLS to real data: CEO compensation and firm performance

We now leave simulated data behind. The question we investigate: **does a CEO's pay depend on how well their company performs?** We measure firm performance by **ROE** (return on equity, a common measure of profitability relative to shareholders' equity) and CEO pay by **salary**.

We use the dataset `ceosal1`, available directly through the `wooldridge` Python package. Everyone in the class works with the exact same 209 observations, collected from *Businessweek* in 1991.


In [ ]:
from wooldridge import load_data

ceo = load_data.data('ceosal1')
ceo.head()


In [ ]:
load_data.data('ceosal1', description=True)


### 3.1 First look and descriptive statistics

**Task 3.1.** Look at the summary statistics for `salary`, `roe`, and `sales` below. In particular, compare the mean and the median (50%) for `salary`: what does a mean much larger than the median suggest about the shape of the distribution?


In [ ]:
ceo[['salary', 'roe', 'sales']].describe()


**Task 3.2.** Plot a histogram of `salary` below. Is the distribution symmetric? Given what you just found in Task 3.1 - and what you saw in Session 1 about linearity in the parameters (logs, in particular) - why might we prefer to work with `lsalary` (the log of salary, already provided in the dataset) rather than `salary` itself in a regression?


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(ceo['salary'], bins=30, color='steelblue')
ax.set_xlabel('CEO salary ($ thousands)')
ax.set_ylabel('count')
ax.set_title('Distribution of CEO salary')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.show()

# TODO: redo the same histogram, but using ceo['lsalary'] instead of ceo['salary']. Compare the two shapes.


### 3.2 A judgment call

Before regressing, let's look at the most extreme observations in the sample.


In [ ]:
ceo.sort_values('salary', ascending=False).head(3)[['salary', 'roe', 'sales', 'finance']]


**Task 3.3 (discussion - there is no single right answer).**
1. Purely *statistically*, would a rule such as "drop any observation more than 3 standard deviations from the mean" flag the top-paid CEO above? Check it by completing the code cell below.
2. *Economically*, is there a good reason to think this observation does not belong in our sample (e.g., a data-entry error, or a firm from a completely different context), or is it simply a legitimately very highly-paid CEO that we should keep?
3. For the rest of this TD we keep the full sample. In one sentence, explain why a purely statistical rule is not, by itself, a sufficient reason to drop an observation.


In [ ]:
z_scores = (ceo['salary'] - ceo['salary'].mean()) / ceo['salary'].std()

# TODO: count how many observations have |z_score| > 3
n_outliers = ...
print(n_outliers, "observation(s) flagged by the 3-standard-deviation rule.")


### 3.3 Does ROE explain (log) salary?

So far, in Sessions 1-2, you derived the OLS estimator and its standard error for a model **without an intercept**: $y_i=\beta x_i + \varepsilon_i$. Real data like this one will basically never pass through the origin - a firm with `roe=0` certainly does not imply a CEO salary of \$0! We therefore need the (very natural) generalization **with an intercept**:
$$
\log(\text{salary}_i) = \beta_0 + \beta_1\, \text{roe}_i + \varepsilon_i.
$$

The logic of least squares is exactly the same as in Session 2 - minimize the sum of squared residuals - except we now search over *two* parameters, $(\beta_0,\beta_1)$, instead of one. Solving the same first-order-condition logic for both parameters gives:
$$
\hat\beta_1 = \frac{\sum_i (x_i-\bar x)(y_i-\bar y)}{\sum_i (x_i - \bar x)^2}, \qquad \hat\beta_0 = \bar y - \hat\beta_1 \bar x,
$$
where $\bar x$ and $\bar y$ are the sample means. (Session 3 will show you how `statsmodels` estimates this - and models with many more regressors - automatically. For now, let's compute it by hand once, to see that it really is the same least-squares idea as before, just centered around the means.)

**Task 3.4.** Complete the formula for `beta1_hat` below.


In [ ]:
roe = ceo['roe'].values
lsalary = ceo['lsalary'].values

roe_bar = roe.mean()
lsalary_bar = lsalary.mean()

# TODO: compute beta1_hat using the centered-sums formula above
beta1_hat = ...
beta0_hat = lsalary_bar - beta1_hat * roe_bar

print("Intercept (beta0_hat):     ", beta0_hat)
print("Slope on roe (beta1_hat):  ", beta1_hat)


Let's now check this against `statsmodels`, which we will use throughout the rest of the course from Session 3 onward:


In [ ]:
import statsmodels.formula.api as smf

res = smf.ols('lsalary ~ roe', data=ceo).fit()
print(res.summary())


**Task 3.5.** Compare `res.params` (printed below) to your hand-computed `beta0_hat` and `beta1_hat`. Do they match?


In [ ]:
print(res.params)


### 3.4 Is the effect statistically significant? Is it economically significant?

**Task 3.6 (statistical reasoning).** Using the regression table above, state $H_0$ and $H_1$ for the coefficient on `roe`, read off the t-statistic and the p-value, and conclude at the 5% level. (This is exactly the t-test from Session 2 - `statsmodels` has simply computed $\hat\beta$, $\text{SE}(\hat\beta)$ and $t$ for you.)

**Task 3.7 (economic reasoning).** In a $\log(\text{salary})\sim\text{roe}$ regression, the coefficient on `roe` is (approximately) a semi-elasticity: a one-point increase in ROE is associated with a $100\times\hat\beta_1$ % change in salary.
1. Compute this percentage.
2. For a CEO earning the *average* salary in this sample, translate this percentage into a dollar amount.
3. In your judgment, is this economically large or small? Justify your answer in 2-3 sentences - there is no single correct answer, but it must be backed by a number.


In [ ]:
pct_effect = 100 * res.params['roe']
print(f"A one-point increase in ROE is associated with a {pct_effect:.2f}% change in salary.")

avg_salary = ceo['salary'].mean()

# TODO: compute the dollar amount corresponding to a 1-point ROE increase, evaluated at the average salary
dollar_effect = ...
print(f"At the average salary (${avg_salary:.0f}k), this represents about ${dollar_effect:.1f}k.")


### 3.5 Bonus (open-ended): looking ahead to Session 3

`ceo` also has a `finance` dummy (`=1` if the firm is a financial firm). Would you expect the ROE-pay relationship to look the same in the financial sector as elsewhere?

**Task 3.8 (bonus).** Compare the average `salary` and `roe` for `finance==1` versus `finance==0` firms (e.g., with `.groupby('finance')`). Comment. We will see in Session 3 how to test this formally, by adding the `finance` dummy - and its interaction with `roe` - directly into the regression.
